# Strategy Pattern
## Design Patterns for the AI Era · Part 3

> **Bad Code → Better Design → Working Python → AI Application**

Swap algorithms at runtime. No if/elif. Perfect for RAG retrieval.

## 1. The Problem

Your RAG system needs multiple retrieval strategies. The naive approach puts them all in one function.

In [ ]:
# ❌ BAD CODE

DOCS = [
    'The Factory pattern centralises object creation.',
    'The Strategy pattern defines a family of algorithms.',
    'The Observer pattern uses events to notify listeners.',
    'High cohesion and low coupling are key design goals.',
    'Python decorators are a form of the Decorator pattern.',
]

def search_bad(query: str, method: str, docs: list[str]) -> list[str]:
    """Problem: every new strategy requires editing this function."""
    if method == 'keyword':
        return [d for d in docs if query.lower() in d.lower()]
    elif method == 'prefix':
        return [d for d in docs if d.lower().startswith(query.lower())]
    elif method == 'exact':
        return [d for d in docs if query == d]
    else:
        raise ValueError(f'Unknown method: {method}')

results = search_bad('pattern', 'keyword', DOCS)
print('Results:', results)

## 2. What's Wrong?

- Adding a new strategy requires editing the existing function
- Cannot test strategies in isolation
- Cannot compose strategies (e.g., run both and merge results)
- The function grows indefinitely as requirements change

## 3. The Pattern

```
Context (RAGRetriever)
    │
    └── strategy: SearchStrategy
                      ├── KeywordSearch
                      ├── VectorSearch
                      └── HybridSearch
```

The context delegates to the strategy. Strategies are interchangeable.

In [ ]:
# ✅ GOOD CODE — Strategy Pattern
from abc import ABC, abstractmethod
import math

class SearchStrategy(ABC):
    """Abstract strategy — the interface."""

    @abstractmethod
    def search(self, query: str, docs: list[str]) -> list[str]:
        ...


class KeywordSearch(SearchStrategy):
    def search(self, query: str, docs: list[str]) -> list[str]:
        q = query.lower()
        return [d for d in docs if q in d.lower()]


class WordCountSearch(SearchStrategy):
    """Rank by how many query words appear in the document."""

    def search(self, query: str, docs: list[str]) -> list[str]:
        words = set(query.lower().split())
        scored = [
            (sum(1 for w in words if w in d.lower()), d)
            for d in docs
        ]
        return [d for score, d in sorted(scored, reverse=True) if score > 0]


class TFIDFSearch(SearchStrategy):
    """Simple TF-IDF style scoring (no external dependencies)."""

    def _tf(self, term: str, doc: str) -> float:
        words = doc.lower().split()
        return words.count(term) / len(words) if words else 0

    def _idf(self, term: str, docs: list[str]) -> float:
        n_docs = len(docs)
        containing = sum(1 for d in docs if term in d.lower())
        return math.log((n_docs + 1) / (containing + 1)) + 1

    def search(self, query: str, docs: list[str]) -> list[str]:
        terms = query.lower().split()
        scored = []
        for doc in docs:
            score = sum(self._tf(t, doc) * self._idf(t, docs) for t in terms)
            scored.append((score, doc))
        return [d for s, d in sorted(scored, reverse=True) if s > 0]


# Context — knows nothing about concrete strategies
class RAGRetriever:
    def __init__(self, strategy: SearchStrategy):
        self._strategy = strategy

    def set_strategy(self, strategy: SearchStrategy) -> None:
        self._strategy = strategy

    def retrieve(self, query: str, docs: list[str]) -> list[str]:
        return self._strategy.search(query, docs)


# Run all three strategies on the same query
query = 'design pattern'
retriever = RAGRetriever(KeywordSearch())

for StrategyClass in [KeywordSearch, WordCountSearch, TFIDFSearch]:
    retriever.set_strategy(StrategyClass())
    results = retriever.retrieve(query, DOCS)
    print(f'\n{StrategyClass.__name__}:')
    for r in results:
        print(f'  → {r}')

## 4. Tests

In [ ]:
# ✅ Tests — each strategy tested in isolation

def test_keyword_search_finds_match():
    results = KeywordSearch().search('factory', DOCS)
    assert any('Factory' in r for r in results)
    print('PASS test_keyword_search_finds_match')

def test_keyword_search_no_match():
    results = KeywordSearch().search('zzzzz', DOCS)
    assert results == []
    print('PASS test_keyword_search_no_match')

def test_retriever_uses_injected_strategy():
    class AlwaysEmpty(SearchStrategy):
        def search(self, q, docs):
            return []

    retriever = RAGRetriever(AlwaysEmpty())
    assert retriever.retrieve('anything', DOCS) == []
    print('PASS test_retriever_uses_injected_strategy')

def test_strategy_swap_at_runtime():
    retriever = RAGRetriever(KeywordSearch())
    r1 = retriever.retrieve('factory', DOCS)
    retriever.set_strategy(WordCountSearch())
    r2 = retriever.retrieve('factory pattern', DOCS)
    # Both return results, different ranking
    assert len(r1) > 0
    assert len(r2) > 0
    print('PASS test_strategy_swap_at_runtime')

test_keyword_search_finds_match()
test_keyword_search_no_match()
test_retriever_uses_injected_strategy()
test_strategy_swap_at_runtime()
print('\nAll tests passed ✅')

## 5. AI / Agent Example — Dynamic Chunking

In [ ]:
# 🤖 AI EXAMPLE — Chunking Strategy for RAG indexing
import re
from abc import ABC, abstractmethod

class ChunkingStrategy(ABC):
    @abstractmethod
    def chunk(self, text: str) -> list[str]: ...

class FixedSizeChunker(ChunkingStrategy):
    def __init__(self, size: int = 200, overlap: int = 20):
        self.size = size
        self.overlap = overlap

    def chunk(self, text: str) -> list[str]:
        chunks = []
        i = 0
        while i < len(text):
            chunks.append(text[i:i + self.size])
            i += self.size - self.overlap
        return chunks

class SentenceChunker(ChunkingStrategy):
    def chunk(self, text: str) -> list[str]:
        return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text) if s.strip()]

class ParagraphChunker(ChunkingStrategy):
    def chunk(self, text: str) -> list[str]:
        return [p.strip() for p in text.split('\n\n') if p.strip()]

class Indexer:
    def __init__(self, chunker: ChunkingStrategy):
        self._chunker = chunker

    def index(self, document: str) -> list[str]:
        return self._chunker.chunk(document)

sample_doc = (
    'Design patterns are reusable solutions to common problems. '
    'They were popularised by the Gang of Four in 1994. '
    'Today they are more relevant than ever for AI systems.\n\n'
    'The Factory pattern centralises object creation. '
    'The Strategy pattern enables interchangeable algorithms.'
)

indexer = Indexer(SentenceChunker())
print('SentenceChunker:')
for chunk in indexer.index(sample_doc):
    print(f'  [{len(chunk):3d}] {chunk[:70]}')

indexer = Indexer(ParagraphChunker())
print('\nParagraphChunker:')
for chunk in indexer.index(sample_doc):
    print(f'  [{len(chunk):3d}] {chunk[:70]}')

## 6. Summary

| | Before | After |
|---|---|---|
| Adding strategy | Edit `search()` function | New class, zero existing edits |
| Testing | Must test all strategies together | Each strategy tested in isolation |
| Runtime swap | Impossible | `retriever.set_strategy(new_strategy)` |
| Open/Closed | ❌ | ✅ |